<a href="https://colab.research.google.com/github/Arian-khoushabi/Fire-Detection-Using-Deep-Learning/blob/main/fire_classification_vgg16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# پروژه تشخیص آتش با استفاده از یادگیری عمیقدر این پروژه از VGG11 پیش‌آموزش‌دیده برای طبقه‌بندی تصاویر fire و non_fire استفاده شده و با تکنیک‌های Data Augmentation و EarlyStopping از Overfitting جلوگیری شده است.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import os

In [ ]:
train_transforms = transforms.Compose([transforms.RandomHorizontalFlip(),    transforms.RandomRotation(10),    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])val_test_transforms = transforms.Compose([    transforms.Resize((224, 224)),    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

In [ ]:
train_dataset = datasets.ImageFolder(root='path_to_train', transform=train_transforms)val_dataset = datasets.ImageFolder(root='path_to_val', transform=val_test_transforms)test_dataset = datasets.ImageFolder(root='path_to_test', transform=val_test_transforms)train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class EarlyStopping:    def __init__(self, patience=5, delta=0.001):        self.patience = patience        self.delta = delta        self.best_loss = None        self.counter = 0        self.early_stop = False    def __call__(self, val_loss):        if self.best_loss is None:            self.best_loss = val_loss        elif val_loss > self.best_loss - self.delta:            self.counter += 1            if self.counter >= self.patience:                self.early_stop = True        else:            self.best_loss = val_loss            self.counter = 0

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")model = models.vgg16(pretrained=True)for param in model.features.parameters():    param.requires_grad = Falsemodel.classifier = nn.Sequential(    nn.Linear(25088, 512),    nn.ReLU(),    nn.Dropout(0.3),    nn.Linear(512, 2))model = model.to(device)criterion = nn.CrossEntropyLoss()optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

In [ ]:
num_epochs = 25train_acc_list = []val_acc_list = []early_stopping = EarlyStopping(patience=5)for epoch in range(num_epochs):    model.train()    correct = 0    total = 0    for images, labels in train_loader:        images, labels = images.to(device), labels.to(device)        optimizer.zero_grad()        outputs = model(images)        loss = criterion(outputs, labels)        loss.backward()        optimizer.step()        _, predicted = torch.max(outputs.data, 1)        total += labels.size(0)        correct += (predicted == labels).sum().item()    train_accuracy = correct / total    train_acc_list.append(train_accuracy)    model.eval()    val_loss = 0    correct = 0    total = 0    with torch.no_grad():        for images, labels in val_loader:            images, labels = images.to(device), labels.to(device)            outputs = model(images)            loss = criterion(outputs, labels)            val_loss += loss.item()            _, predicted = torch.max(outputs.data, 1)            total += labels.size(0)            correct += (predicted == labels).sum().item()    val_accuracy = correct / total    val_acc_list.append(val_accuracy)    print(f"Epoch {epoch+1}, Train Acc: {train_accuracy:.4f}, Val Acc: {val_accuracy:.4f}, Val Loss: {val_loss:.4f}")    early_stopping(val_loss)    if early_stopping.early_stop:        print(f"Early stopping at epoch {epoch+1}")        break

In [ ]:
plt.plot(train_acc_list, label='Train Accuracy')plt.plot(val_acc_list, label='Validation Accuracy')plt.xlabel('Epoch')plt.ylabel('Accuracy')plt.legend()plt.title('Accuracy per Epoch')plt.show()

In [ ]:
model.eval()correct = 0total = 0all_preds = []all_labels = []with torch.no_grad():    for images, labels in test_loader:        images, labels = images.to(device), labels.to(device)        outputs = model(images)        _, predicted = torch.max(outputs.data, 1)        total += labels.size(0)        correct += (predicted == labels).sum().item()        all_preds.extend(predicted.cpu().numpy())        all_labels.extend(labels.cpu().numpy())test_accuracy = correct / totalprint(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
cm = confusion_matrix(all_labels, all_preds)disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_dataset.classes)disp.plot(cmap=plt.cm.Blues)plt.title("Confusion Matrix - Test Set")plt.show()